## Customer Segmentation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [6]:
df = pd.read_csv('marketing_campaign.csv', sep='\t')
print("Shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'marketing_campaign.csv'

In [ ]:
df.info()

In [ ]:
print(df.isnull().sum()[df.isnull().sum() > 0])

### Data Cleaning

In [ ]:
# Drop rows with missing Income — only 24 rows so safe to remove
df.dropna(subset=['Income'], inplace=True)

# Drop columns that add no value for clustering
df.drop(columns=['ID', 'Z_CostContact', 'Z_Revenue'], inplace=True)

# Convert Dt_Customer to datetime
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], dayfirst=True)

print("Shape after cleaning:", df.shape)
print("Missing values remaining:", df.isnull().sum().sum())

## 3. Feature Engineering

We create new features that are more meaningful for clustering than the raw columns.

In [ ]:
# Age — more useful than birth year
df['Age'] = 2024 - df['Year_Birth']

# Total amount spent across all product categories
df['Total_Spent'] = (df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] +
                     df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds'])

# Total number of purchases across all channels
df['Total_Purchases'] = (df['NumWebPurchases'] + df['NumCatalogPurchases'] +
                         df['NumStorePurchases'] + df['NumDealsPurchases'])

# How long the customer has been with the company (in days)
df['Days_As_Customer'] = (pd.Timestamp('2024-01-01') - df['Dt_Customer']).dt.days

# Total number of children at home
df['Total_Children'] = df['Kidhome'] + df['Teenhome']

# Total campaigns accepted
df['Total_Campaigns_Accepted'] = (df['AcceptedCmp1'] + df['AcceptedCmp2'] +
                                   df['AcceptedCmp3'] + df['AcceptedCmp4'] +
                                   df['AcceptedCmp5'] + df['Response'])

print("New features added:")
df[['Age', 'Total_Spent', 'Total_Purchases', 'Days_As_Customer',
    'Total_Children', 'Total_Campaigns_Accepted']].describe().round(2)

In [ ]:
# Drop columns we no longer need after feature engineering
df.drop(columns=['Year_Birth', 'Dt_Customer', 'Kidhome', 'Teenhome',
                 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts',
                 'MntSweetProducts', 'MntGoldProds',
                 'NumWebPurchases', 'NumCatalogPurchases',
                 'NumStorePurchases', 'NumDealsPurchases',
                 'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3',
                 'AcceptedCmp4', 'AcceptedCmp5', 'Response'], inplace=True)

print("Shape:", df.shape)
print("Remaining columns:", df.columns.tolist())

## 4. Exploratory Data Analysis

We explore the data to understand distributions and patterns before clustering.

In [ ]:
# Age distribution
plt.figure(figsize=(8, 4))
df['Age'].hist(bins=30, color='steelblue', edgecolor='black', alpha=0.8)
plt.title('Age Distribution of Customers')
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Income distribution
plt.figure(figsize=(8, 4))
df['Income'].hist(bins=30, color='coral', edgecolor='black', alpha=0.8)
plt.title('Income Distribution of Customers')
plt.xlabel('Income')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Total Spent vs Income
plt.figure(figsize=(8, 5))
plt.scatter(df['Income'], df['Total_Spent'], alpha=0.4, color='steelblue')
plt.xlabel('Income')
plt.ylabel('Total Spent')
plt.title('Income vs Total Spent')
plt.tight_layout()
plt.show()

In [ ]:
# Education breakdown
plt.figure(figsize=(7, 4))
df['Education'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Education Level Distribution')
plt.xlabel('Education')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
numeric_df = df.select_dtypes(include=['int64', 'float64'])
sns.heatmap(numeric_df.corr().round(2), annot=True, cmap='coolwarm',
            fmt='.2f', linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 5. Preprocessing

### 5.1 Encoding Categorical Columns

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

le = LabelEncoder()
df['Education'] = le.fit_transform(df['Education'].astype(str))
df['Marital_Status'] = le.fit_transform(df['Marital_Status'].astype(str))

print("Encoding done.")
df.head()

### 5.2 Feature Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print("Scaled shape:", X_scaled.shape)

### 5.3 Dimensionality Reduction — PCA

In [ ]:
# Check how many components explain 95% of variance
pca_full = PCA()
pca_full.fit(X_scaled)

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(pca_full.explained_variance_ratio_) + 1),
         np.cumsum(pca_full.explained_variance_ratio_),
         marker='o', color='steelblue', linewidth=2)
plt.axhline(y=0.95, color='red', linestyle='--', label='95% Variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — How Many Components Do We Need?')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Apply PCA keeping 95% of variance
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

print(f"Original features : {X_scaled.shape[1]}")
print(f"After PCA         : {X_pca.shape[1]}")
print(f"Variance retained : {pca.explained_variance_ratio_.sum()*100:.2f}%")

In [ ]:
# 2D projection for visualisation
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(8, 5))
plt.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.4, color='steelblue', edgecolors='none')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA — 2D View of Customer Data')
plt.tight_layout()
plt.show()

## 6. K-Means Clustering

K-Means groups customers by finding K cluster centres and assigning each customer to the nearest one. We use the Elbow Method to find the best K.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Elbow Method — plot WCSS for K = 2 to 10
wcss = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_pca)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(k_range, wcss, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.title('Elbow Method — Finding the Optimal K')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette scores to confirm best K
print("Silhouette Scores:")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_pca)
    score = silhouette_score(X_pca, labels)
    print(f"  K={k} : {score:.4f}")

In [ ]:
# Apply K-Means with the best K
best_k = 4
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_pca)

sil = silhouette_score(X_pca, kmeans_labels)
print(f"K-Means Silhouette Score (K={best_k}): {sil:.4f}")

In [ ]:
# Visualise K-Means clusters
plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1],
                      c=kmeans_labels, cmap='viridis',
                      alpha=0.5, edgecolors='none')
plt.colorbar(scatter, label='Cluster')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title(f'K-Means Clusters (K={best_k})')
plt.tight_layout()
plt.show()

## 7. DBSCAN Clustering

DBSCAN finds clusters based on density. It does not need us to specify the number of clusters in advance and can detect outlier customers as noise points.

In [ ]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=0.5, min_samples=5)
db_labels = db.fit_predict(X_pca)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)

print(f"Clusters found : {n_clusters}")
print(f"Noise points   : {n_noise}")

# Silhouette score (excluding noise)
mask = db_labels != -1
if mask.sum() > 1 and len(set(db_labels[mask])) > 1:
    sil_db = silhouette_score(X_pca[mask], db_labels[mask])
    print(f"Silhouette Score : {sil_db:.4f}")

In [ ]:
# Visualise DBSCAN clusters
plt.figure(figsize=(9, 6))
unique_labels = set(db_labels)
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))

for k, col in zip(unique_labels, colors):
    if k == -1:
        col = 'black'
        label = 'Noise'
    else:
        label = f'Cluster {k}'
    mask = db_labels == k
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                c=[col], alpha=0.5, label=label, edgecolors='none')

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('DBSCAN Clusters')
plt.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Cluster Interpretation

We add the K-Means cluster labels back to the original dataframe and analyse what makes each cluster unique.

In [ ]:
# Add cluster labels to original dataframe
df['Cluster'] = kmeans_labels

# Summary stats per cluster
cluster_summary = df.groupby('Cluster')[['Age', 'Income', 'Total_Spent',
                                          'Total_Purchases', 'Total_Children',
                                          'Total_Campaigns_Accepted']].mean().round(2)
cluster_summary

In [ ]:
# Visualise key features across clusters
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

features = ['Age', 'Income', 'Total_Spent',
            'Total_Purchases', 'Total_Children', 'Total_Campaigns_Accepted']
colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']

for ax, feature in zip(axes.flatten(), features):
    for cluster in range(best_k):
        data = df[df['Cluster'] == cluster][feature]
        ax.hist(data, bins=20, alpha=0.5, label=f'Cluster {cluster}',
                color=colors[cluster], edgecolor='none')
    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)

plt.suptitle('Feature Distribution by Cluster', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Average Total Spent per cluster — bar chart
plt.figure(figsize=(8, 5))
cluster_summary['Total_Spent'].plot(kind='bar', color=colors, edgecolor='black')
plt.title('Average Total Spent by Cluster')
plt.xlabel('Cluster')
plt.ylabel('Average Total Spent')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Conclusion

| Cluster | Characteristics |
|---|---|
| Cluster 0 | To be filled after running |
| Cluster 1 | To be filled after running |
| Cluster 2 | To be filled after running |
| Cluster 3 | To be filled after running |

**Key takeaways:**
- K-Means successfully grouped customers into distinct segments based on spending, income, age and purchasing behaviour
- PCA helped reduce the feature space and made the clusters easier to visualise
- DBSCAN identified outlier customers that do not fit neatly into any cluster
- Each cluster represents a different type of customer that the business can target differently